# Topic 4 participant practical: persistent homology as homology with memory

This investigation begins with a small class that dies under inclusion, then reproduces the A/B/C barcode and persistence diagram from the reader.

Replace the `abc` array with another $n\times d$ point cloud to run the same Rips persistence calculation on new data. We move through

$$K_a\subseteq K_b\longrightarrow H_p(K_a;\mathbb F_2)\to H_p(K_b;\mathbb F_2)
\longrightarrow\text{persistence module}\longrightarrow\text{barcode}.$$

The hand calculations come first. A library calculation is used only after the spaces, maps and interval summary have been identified.

The glossary defines *induced map*, *birth*, *death*, *persistence module*, *barcode* and *essential class*.

**Working rule.** Run one section at a time. Before each TODO, state what shape, dimension or direction you expect in the output. Optional extensions come only after the core checkpoints agree.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG=np.random.default_rng(3024)

def rank_mod2(A):
    A=np.array(A,dtype=np.uint8,copy=True)%2; row=rank=0
    for col in range(A.shape[1]):
        piv=np.flatnonzero(A[row:,col])
        if not len(piv): continue
        q=row+piv[0]; A[[row,q]]=A[[q,row]]
        for i in range(A.shape[0]):
            if i!=row and A[i,col]: A[i]^=A[row]
        row+=1; rank+=1
        if row==A.shape[0]: break
    return rank

def plot_barcode(diagrams, titles=('H0','H1')):
    fig,axes=plt.subplots(1,len(diagrams),figsize=(10,3.4))
    for dim,(ax,D) in enumerate(zip(np.atleast_1d(axes),diagrams)):
        finite=D[np.isfinite(D[:,1])] if len(D) else D
        cap=max(finite[:,1].max() if len(finite) else 1, D[:,0].max() if len(D) else 1)*1.08
        for y,(b,d) in enumerate(D): ax.hlines(y,b,cap if np.isinf(d) else d,lw=2.5)
        ax.set_title(titles[dim]); ax.set_xlabel('Rips distance threshold ε'); ax.set_yticks([]); ax.set_xlim(left=0)
    plt.tight_layout(); plt.show()

print('Using coefficients in F_2.')

## 1. Observe: participant checkpoint

Let $X$ be the outline triangle and $Y$ the same vertices and edges plus the triangular face. The inclusion $X\hookrightarrow Y$ preserves every old simplex and chain.

The question is not whether the old edge cycle still exists as a chain. It does. The question is what happens to its class after $Y$ supplies a new 2-chain.

## 2. Predict: participant checkpoint

Before computing:

1. Predict $H_1(X;\mathbb F_2)$ and $H_1(Y;\mathbb F_2)$.
2. Is the induced map $H_1(X)\to H_1(Y)$ injective?
3. Two vertices merge after an edge is added. Which direction in $H_0\cong\mathbb F_2^2$ is killed?
4. If two unrelated classes live on $[1,4)$ and $[2,6)$, what is the dimension of the module at parameters 0, 1.5, 3 and 5?

## 3. Implement: participant checkpoint

### A. A class dies when it becomes a boundary

The cycle space of the three-edge outline is one-dimensional. Compare the image of $\partial_2$ before and after the face is added.

In [ ]:
d2_X=np.zeros((3,0),dtype=np.uint8)
d2_Y=np.ones((3,1),dtype=np.uint8)
dim_Z1=1
beta1_X=dim_Z1-rank_mod2(d2_X)
beta1_Y=dim_Z1-rank_mod2(d2_Y)
print('beta_1(X)=',beta1_X,'beta_1(Y)=',beta1_Y)

### B. Components merge through a non-injective map

In bases $(e_1,e_2)$ before the edge and $(e)$ afterwards, the induced map is represented by $[1\ 1]$.

In [ ]:
H0_map=np.array([[1,1]],dtype=np.uint8)
for v in [np.array([1,0]),np.array([0,1]),np.array([1,1])]:
    print(v,'maps to',(H0_map@v)%2)

## 4. Compare: participant checkpoint

### A. Betti counts versus module maps

Two modules can have the same dimension at every sampled parameter but connect their vector spaces differently. A sequence of Betti numbers records only vertical slice sizes. The persistence module retains the compatible maps.

For the two intervals $[1,4)$ and $[2,6)$, count how many bars cross each requested parameter.

In [ ]:
bars=[(1.,4.,'A'),(2.,6.,'B')]
fig,ax=plt.subplots(figsize=(8,2.8))
for y,(b,d,label) in enumerate(bars):
    ax.hlines(y,b,d,lw=5); ax.text(d+.12,y,label,va='center')
ax.set_xlim(0,7); ax.set_yticks([]); ax.set_xlabel('filtration parameter'); ax.set_title('Two interval summands'); plt.show()

def dimension_at(a): return sum(b<=a<d for b,d,_ in bars)
for a in [0,1.5,3,5]: print(a,dimension_at(a))

### Triangle filtration checkpoint

Consider the order $v_0,v_1,e_{01},v_2,e_{12},e_{02},f_{012}$.

| Addition | Predict the event |
|---|---|
| $v_0,v_1,v_2$ | |
| $e_{01},e_{12}$ | |
| $e_{02}$ | |
| $f_{012}$ | |

Name the two edge-component pairings and the edge-face pairing. Explain why the face kills a class rather than deleting its edge-cycle chain.

### B. From hand calculation to the A/B/C barcode and diagram

We now use `ripser` on the A/B/C locations from the reader. `ripser` reports the Rips **pairwise-distance threshold** $\varepsilon$, so $\varepsilon=2r$ relative to Topic 3's ball-radius convention.

Before running the cell, predict when the local B-scale loops and the larger A-scale loop should appear.

In [ ]:
from pathlib import Path

candidates=[Path('data/abc_points.csv'),Path('../../data/abc_points.csv')]
abc_path=next((path for path in candidates if path.exists()),None)
if abc_path is not None:
    abc=np.loadtxt(abc_path,delimiter=',',skiprows=1)
else:
    inner=np.array([[0,0],[0,22],[0,44],[0,66],[0,88],[22,0],[42,5],[50,23],[22,44],[43,44],[50,65],[42,85],[22,88]],float)
    outer=np.array([[350,35,.82],[292,105,.82],[408,105,.82],[235,183,.82],[465,183,.82],[182,263,.82],[518,263,.82],[300,225,.78],[400,225,.78],[125,340,.82],[575,340,.82]],float)
    abc=np.vstack([inner*scale+[x,y] for x,y,scale in outer])

plt.figure(figsize=(6,4.2))
plt.scatter(abc[:,0],-abc[:,1],s=11,color='#267f82')
plt.gca().set_aspect('equal');plt.axis('off');plt.title('A/B/C locations')
plt.show()

In [ ]:
from ripser import ripser

D_abc=ripser(abc,maxdim=1)['dgms'][1]
finite=D_abc[np.isfinite(D_abc[:,1])]
plot_barcode([finite],titles=('H1',))
limit=finite[:,1].max()*1.06
plt.figure(figsize=(4.5,4.2))
plt.plot([0,limit],[0,limit],'--',color='#819092')
plt.scatter(finite[:,0],finite[:,1],s=28,color='#267f82')
plt.xlim(0,limit);plt.ylim(0,limit);plt.gca().set_aspect('equal')
plt.xlabel('birth ε');plt.ylabel('death ε');plt.title('H1 persistence diagram')
plt.show()
print('finite H1 intervals:',len(finite))
print('longest interval:',finite[np.argmax(finite[:,1]-finite[:,0])])
# TODO: identify the cluster of B-scale intervals and the later A-scale interval.

## 5. Interpret: participant checkpoint

1. Why can inclusion of complexes induce a non-injective map on homology?
2. What information do module maps retain that Betti numbers discard?
3. Under what one-parameter finiteness or tameness conditions is a barcode a complete interval summary?
4. What does an infinite death mean in this computed filtration, and when could it instead reflect truncation?
5. How do the local B-scale intervals and global A-scale interval appear differently in the barcode and diagram?
6. What information about the location of a loop is missing from both displays?

**† Qualification.** This is persistence of the A/B/C point locations under the chosen Rips construction, not recognition of the printed letters.